# Analyse des Concentrations de Planctons Marins
## Jalon 1/3 : Exploration des Données & Classification Baseline


---

### Résumé
Ce notebook constitue le premier jalon du projet d'analyse des concentrations de planctons marins.
Il couvre l'exploration des données SEANOE (Panaïotis et al., 2023), la transformation de Hellinger,
l'Analyse en Composantes Principales (ACP) et la classification baseline par K-means.


## Table des matières

1. [Présentation des Données et Exploration](#partie-1)
   - 1.1 Chargement et structure des données
   - 1.2 Analyse des valeurs manquantes
   - 1.3 Statistiques descriptives
   - 1.4 Visualisations exploratoires
   - 1.5 Résumé de l'article de référence
2. [Reproduction de l'Analyse de Référence (Baseline)](#partie-2)
   - 2.1 Transformation de Hellinger
   - 2.2 Analyse en Composantes Principales (ACP)
   - 2.3 Classification K-means
   - 2.4 Interprétation des résultats
3. [Sauvegarde des résultats](#sauvegarde)
4. [Conclusions & Perspectives](#conclusions)


In [ ]:
# === Imports ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Paramètres graphiques globaux
plt.rcParams.update({
    'figure.dpi': 100,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.facecolor': 'white',
})
sns.set_style('whitegrid')
PALETTE = 'Set2'
RANDOM_STATE = 42
print('Imports OK')

---
# Partie 1 : Présentation des Données et Exploration

## 1.1 Contexte Scientifique

Le **plancton marin** constitue la base de la chaîne alimentaire océanique et joue un rôle central dans les cycles biogéochimiques globaux, notamment le cycle du carbone. Les communautés planctoniques présentent une grande diversité de groupes taxonomiques (copépodes, appendiculaires, chaetognathes, etc.) dont la distribution spatiale et temporelle est déterminée par les conditions physico-chimiques de l'environnement (température, salinité, disponibilité en nutriments).

### Source des données
Ce projet s'appuie sur le jeu de données **SEANOE** (SEA scieNtific Open data Edition), référencé par Panaïotis et al. (2023). Les données ont été collectées *in situ* entre **2008 et 2019** sur une profondeur de **0 à 500 mètres**. Chaque observation correspond à un échantillon identifié par `psampleid` et contient :
- Des **concentrations** de groupes planctoniques (organismes/volume)
- Des **variables environnementales** (température, salinité, chlorophylle, etc.)
- Des **métadonnées contextuelles** (position géographique, date, couche océanique)

### Objectif du jalon 1
Reproduire l'analyse de référence de Panaïotis et al. (2023) :
1. Explorer et comprendre les données
2. Appliquer la transformation de Hellinger sur les concentrations
3. Réaliser une ACP pour réduire la dimensionnalité
4. Classifier les communautés planctoniques (baseline K-means)

In [ ]:
# === Chargement des données ===
DATA_PATH = 'data/103673.csv'

df = pd.read_csv(DATA_PATH, na_values=['NA'])

# Conversion de la colonne datetime
df['datetime'] = pd.to_datetime(df['datetime'])
df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month

print(f'=== Aperçu du jeu de données ===')
print(f'Dimensions : {df.shape[0]} observations x {df.shape[1]} colonnes')
print(f'Période : {df["datetime"].min().date()} → {df["datetime"].max().date()}')
print(f'Couverture géographique : lat [{df["lat"].min():.2f}, {df["lat"].max():.2f}] | lon [{df["lon"].min():.2f}, {df["lon"].max():.2f}]')
df.head(3)

In [ ]:
# === Identification des groupes de colonnes ===

COLS_CONTEXT = ['psampleid', 'lat', 'lon', 'datetime', 'day_night', 'prod', 'layer']

# Colonnes de concentration : entre 'layer' et 'temp'
idx_layer = df.columns.get_loc('layer')
idx_temp  = df.columns.get_loc('temp')
COLS_CONC = df.columns[idx_layer + 1 : idx_temp].tolist()

# Colonnes environnementales : de 'temp' à 'kd490' (avant les colonnes year/month ajoutées)
idx_kd490 = df.columns.get_loc('kd490')
COLS_ENV  = df.columns[idx_temp : idx_kd490 + 1].tolist()

print(f'Colonnes contextuelles ({len(COLS_CONTEXT)}) : {COLS_CONTEXT}')
print(f'\nColonnes de concentration ({len(COLS_CONC)}) :')
print(COLS_CONC)
print(f'\nColonnes environnementales ({len(COLS_ENV)}) :')
print(COLS_ENV)

In [ ]:
# === Analyse des valeurs manquantes ===

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Valeurs manquantes': missing,
    'Pourcentage (%)': missing_pct
}).query('`Valeurs manquantes` > 0').sort_values('Pourcentage (%)', ascending=False)

print('=== Valeurs manquantes ===')
if missing_df.empty:
    print('Aucune valeur manquante !')
else:
    print(missing_df.to_string())
    print(f'\nConcentrations : {df[COLS_CONC].isnull().sum().sum()} valeurs manquantes')
    print(f'Variables env.  : {df[COLS_ENV].isnull().sum().sum()} valeurs manquantes')

# Préparation du jeu de données nettoyé (pour les analyses environnementales)
df_clean = df.dropna(subset=COLS_ENV).copy()
print(f'\nAprès suppression des NA env : {df_clean.shape[0]} observations ({len(df) - df_clean.shape[0]} supprimées)')

# Valeurs uniques des variables catégorielles
print(f"\nVariables catégorielles :")
for col in ['day_night', 'prod', 'layer']:
    print(f"  {col} : {sorted(df[col].dropna().unique().tolist())}")

## 1.2 Statistiques Descriptives

In [ ]:
# === Statistiques descriptives — Concentrations ===

stats_conc = df[COLS_CONC].describe().T
stats_conc['skewness'] = df[COLS_CONC].skew()
stats_conc['non_zero_%'] = (df[COLS_CONC] > 0).mean() * 100
stats_conc = stats_conc[['count', 'mean', 'std', 'min', '50%', 'max', 'skewness', 'non_zero_%']]
stats_conc.columns = ['N', 'Moyenne', 'Écart-type', 'Min', 'Médiane', 'Max', 'Asymétrie', 'Présence (%)']

print('=== Statistiques descriptives — Variables de concentration ===')
print(stats_conc.round(5).to_string())

# Groupes les plus fréquemment observés
print(f'\nTop 5 groupes les plus présents :')
top5 = stats_conc['Présence (%)'].nlargest(5)
for name, val in top5.items():
    print(f'  {name:30s} : {val:.1f}%')

In [ ]:
# === Statistiques descriptives — Variables environnementales ===

env_descriptions = {
    'temp': 'Température (°C)',
    'sal': 'Salinité (PSU)',
    'sigma': 'Densité (kg/m³)',
    'chla': 'Chlorophylle a (mg/m³)',
    'oxy_ml': 'Oxygène dissous (ml/L)',
    'aou': 'AOU (ml/L)',
    'snow_conc': 'Marine snow (conc.)',
    'bulk_conc': 'Concentration totale',
    'bulk_biov': 'Biovolume total',
    'thermo': 'Thermocline (m)',
    'halo': 'Halocline (m)',
    'mld': 'Mixed Layer Depth (m)',
    'pyc': 'Pycnocline (m)',
    'dcm': 'Deep Chlorophyll Maximum (m)',
    'ze': 'Zone euphotique (m)',
    'strat': 'Stratification',
    'epi': 'Profondeur épipélagique (m)',
    'chla_surf': 'Chla surface (mg/m³)',
    'bbp': 'Rétrodiffusion (m⁻¹)',
    'poc': 'POC (mg/m³)',
    'pic': 'PIC (mol/m³)',
    'par': 'Rayonnement (μmol/m²/s)',
    'kd490': 'Atténuation lumineuse (m⁻¹)',
}

stats_env = df[COLS_ENV].describe().T[['mean', 'std', 'min', '50%', 'max']]
stats_env.columns = ['Moyenne', 'Écart-type', 'Min', 'Médiane', 'Max']
stats_env.index = [env_descriptions.get(c, c) for c in COLS_ENV]

print('=== Statistiques descriptives — Variables environnementales ===')
print(stats_env.round(4).to_string())

## 1.3 Visualisations Exploratoires

In [ ]:
# === Distributions des concentrations ===

# Trier les groupes par présence décroissante
presence = (df[COLS_CONC] > 0).mean() * 100
sorted_conc = presence.sort_values(ascending=False)

fig, axes = plt.subplots(4, 7, figsize=(22, 14))
axes = axes.flatten()

colors = plt.cm.tab20(np.linspace(0, 1, len(COLS_CONC)))

for i, col in enumerate(sorted_conc.index):
    ax = axes[i]
    data_nonzero = df[col][df[col] > 0]
    if len(data_nonzero) > 0:
        ax.hist(np.log10(data_nonzero + 1e-10), bins=30, color=colors[i], alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.set_title(f'{col}\n({sorted_conc[col]:.0f}% présent)', fontsize=7.5, pad=2)
    ax.set_xlabel('log10(conc.)', fontsize=7)
    ax.tick_params(labelsize=7)

# Masquer les axes vides
for j in range(len(COLS_CONC), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distributions des concentrations planctoniques (log₁₀, valeurs > 0)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_01_distributions_concentrations.png', dpi=120, bbox_inches='tight')
plt.show()
print('Les distributions sont très asymétriques (right-skewed) — la transformation de Hellinger est justifiée.')

In [ ]:
# === Distributions des variables environnementales clés ===

key_env = ['temp', 'sal', 'chla', 'oxy_ml', 'mld', 'ze', 'poc', 'par']
key_labels = ['Température (°C)', 'Salinité (PSU)', 'Chlorophylle a (mg/m³)',
              'Oxygène dissous (ml/L)', 'MLD (m)', 'Zone euphotique (m)',
              'POC (mg/m³)', 'PAR (μmol/m²/s)']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, (col, label) in enumerate(zip(key_env, key_labels)):
    data = df[col].dropna()
    axes[i].hist(data, bins=40, color='steelblue', alpha=0.75, edgecolor='white', linewidth=0.3)
    axes[i].axvline(data.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Moy. = {data.mean():.2f}')
    axes[i].axvline(data.median(), color='orange', linestyle=':', linewidth=1.5, label=f'Méd. = {data.median():.2f}')
    axes[i].set_title(label, fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=8)
    axes[i].tick_params(labelsize=9)

fig.suptitle('Distributions des variables environnementales clés', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_02_distributions_environnement.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# === Carte spatiale des observations ===

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# --- Sous-figure 1 : localisation par couche océanique ---
layers = df['layer'].dropna().unique()
colors_layer = {'epi': '#2196F3', 'meso': '#FF9800', 'bathy': '#4CAF50', 'abysso': '#9C27B0'}
default_colors = plt.cm.Set1(np.linspace(0, 1, len(layers)))

for k, layer in enumerate(sorted(layers)):
    mask = df['layer'] == layer
    c = colors_layer.get(layer, default_colors[k])
    axes[0].scatter(df.loc[mask, 'lon'], df.loc[mask, 'lat'],
                    s=8, alpha=0.5, color=c, label=layer)

axes[0].set_xlabel('Longitude (°)', fontweight='bold')
axes[0].set_ylabel('Latitude (°)', fontweight='bold')
axes[0].set_title('Distribution spatiale par couche océanique', fontweight='bold')
axes[0].legend(title='Couche', markerscale=2)
axes[0].grid(True, alpha=0.3)

# --- Sous-figure 2 : localisation par année ---
years = sorted(df['year'].dropna().unique())
cmap = plt.cm.viridis
norm = mcolors.Normalize(vmin=min(years), vmax=max(years))
sc = axes[1].scatter(df['lon'], df['lat'],
                     s=8, alpha=0.5,
                     c=df['year'], cmap=cmap, norm=norm)
plt.colorbar(sc, ax=axes[1], label='Année')
axes[1].set_xlabel('Longitude (°)', fontweight='bold')
axes[1].set_ylabel('Latitude (°)', fontweight='bold')
axes[1].set_title('Distribution spatiale par année (2008–2019)', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Distribution géographique des échantillons', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_03_carte_spatiale.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Nombre total de stations : {len(df)}')
print(f'Couverture géographique : {df["lat"].min():.1f}°N → {df["lat"].max():.1f}°N | {df["lon"].min():.1f}°E → {df["lon"].max():.1f}°E')

In [ ]:
# === Distribution temporelle des observations ===

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Nombre d'observations par année
obs_year = df.groupby('year').size()
axes[0].bar(obs_year.index, obs_year.values, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Année', fontweight='bold')
axes[0].set_ylabel('Nombre d\'observations', fontweight='bold')
axes[0].set_title('Observations par année', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Nombre d'observations par mois
month_names = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']
obs_month = df.groupby('month').size().reindex(range(1, 13), fill_value=0)
axes[1].bar(range(1, 13), obs_month.values, color='coral', edgecolor='white', alpha=0.85)
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(month_names, rotation=45)
axes[1].set_xlabel('Mois', fontweight='bold')
axes[1].set_ylabel('Nombre d\'observations', fontweight='bold')
axes[1].set_title('Observations par mois (saisonnalité)', fontweight='bold')

# Répartition jour/nuit
dn_counts = df['day_night'].value_counts()
axes[2].pie(dn_counts.values, labels=dn_counts.index, autopct='%1.1f%%',
            colors=['#FFD54F', '#5C6BC0'], startangle=90, textprops={'fontsize': 12})
axes[2].set_title('Répartition jour / nuit', fontweight='bold')

plt.suptitle('Distribution temporelle des observations (2008–2019)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_04_distribution_temporelle.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# === Matrice de corrélation — concentrations (présence > 20%) ===

# Sélectionner les groupes présents dans plus de 20% des échantillons
presence_mask = (df[COLS_CONC] > 0).mean() > 0.20
cols_frequent = df[COLS_CONC].columns[presence_mask].tolist()

corr_conc = df[cols_frequent].corr(method='spearman')  # Spearman car données non-normales

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# Corrélations entre concentrations fréquentes
mask_upper = np.triu(np.ones_like(corr_conc, dtype=bool))
sns.heatmap(corr_conc, mask=mask_upper, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1, ax=axes[0],
            annot_kws={'size': 8}, linewidths=0.3)
axes[0].set_title(f'Corrélations de Spearman — Concentrations\n(groupes présents > 20%, n={len(cols_frequent)})',
                  fontweight='bold')
axes[0].tick_params(axis='x', rotation=45, labelsize=9)
axes[0].tick_params(axis='y', labelsize=9)

# Corrélations entre variables environnementales clés
key_env_corr = ['temp', 'sal', 'chla', 'oxy_ml', 'mld', 'ze', 'poc', 'par', 'kd490']
corr_env = df[key_env_corr].corr(method='spearman')
mask_env = np.triu(np.ones_like(corr_env, dtype=bool))
sns.heatmap(corr_env, mask=mask_env, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1, ax=axes[1],
            annot_kws={'size': 9}, linewidths=0.3)
axes[1].set_title('Corrélations de Spearman — Variables environnementales clés',
                  fontweight='bold')
axes[1].tick_params(axis='x', rotation=45, labelsize=9)
axes[1].tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig('fig_05_correlations.png', dpi=120, bbox_inches='tight')
plt.show()

## 1.4 Résumé de l'Article de Référence

**Panaïotis, T. et al. (2023).** *Content-aware segmentation of objects spanning a large size range: application to plankton images from the Mediterranean Sea.*

### Méthodologie de l'article

| Étape | Méthode utilisée | Détails |
|-------|-------------------|---------|
| Prétraitement | **Transformation de Hellinger** | $y'_{ij} = \sqrt{y_{ij} / y_{i+}}$ où $y_{i+}$ est la somme totale par échantillon |
| Réduction dim. | **ACP** | Sur données transformées, centralisation automatique |
| Classification | **K-means** | Nombre de clusters déterminé par critère du coude |
| Validation | **Cohérence écologique** | Interprétation des clusters via variables environnementales |

### Justification de la transformation de Hellinger
La transformation de Hellinger est particulièrement adaptée aux données de composition (concentrations) car :
- Elle normalise les données par la somme totale → compare les **profils** planctoniques, pas les abondances absolues
- Elle réduit l'effet des valeurs extrêmes (double racine carrée implicite)
- Elle permet l'utilisation de la **distance euclidienne** (qui correspond alors à la distance de Hellinger, adaptée aux données écologiques)
- Propriété de vérification : $\sum_j (y'_{ij})^2 = 1$ pour tout échantillon $i$ non nul

## 1.5 Problématique et Questions de Recherche

**Question principale** : Comment se structurent les communautés planctoniques de la Mer Méditerranée et comment évoluent-elles dans l'espace et le temps ?

**Questions secondaires** :
1. Combien de communautés planctoniques distinctes peut-on identifier dans les données ?
2. Quelles variables environnementales expliquent le mieux la structuration de ces communautés ?
3. Existe-t-il une saisonnalité ou un gradient spatial dans la distribution des communautés ?
4. Les méthodes non-linéaires révèlent-elles des structures cachées non détectées par l'ACP ?

---
# Partie 2 : Reproduction de l'Analyse de Référence (Baseline)

## 2.1 Transformation de Hellinger

La transformation est définie par :

$$y'_{ij} = \sqrt{\frac{y_{ij}}{y_{i+}}}$$

où $y_{ij}$ est la concentration du groupe $j$ dans l'échantillon $i$, et $y_{i+} = \sum_j y_{ij}$ est la somme totale des concentrations pour cet échantillon.

In [ ]:
# === Transformation de Hellinger ===

def hellinger_transform(df_conc):
    """
    Applique la transformation de Hellinger (1909) sur les données de concentration.

    Formule : y'_ij = sqrt(y_ij / y_i+)
    où y_i+ = somme des concentrations de l'échantillon i.

    Propriété : sum_j (y'_ij)^2 = 1 pour les échantillons non nuls.

    Paramètres
    ----------
    df_conc : DataFrame (n_samples x n_species)

    Retourne
    --------
    DataFrame transformé de même forme
    """
    row_sums = df_conc.sum(axis=1)
    # Éviter la division par zéro (échantillons vides → rester à 0)
    row_sums_safe = row_sums.replace(0, np.nan)
    # Normalisation par la somme → proportions
    df_prop = df_conc.div(row_sums_safe, axis=0)
    # Racine carrée des proportions
    df_hell = np.sqrt(df_prop.fillna(0))
    return df_hell


# Application sur les données de concentration
df_hellinger = hellinger_transform(df[COLS_CONC])

print('=== Résultat de la transformation de Hellinger ===')
print(f'Shape : {df_hellinger.shape}')
print(f'Valeurs min / max : {df_hellinger.min().min():.6f} / {df_hellinger.max().max():.6f}')
print(f'Valeurs manquantes : {df_hellinger.isnull().sum().sum()}')
print('\nAperçu (3 premières lignes) :')
df_hellinger.head(3)

In [ ]:
# === Vérification de la transformation de Hellinger ===

# Propriété : sum_j (y'_ij)^2 = 1 pour tout échantillon non nul
sum_squares = (df_hellinger ** 2).sum(axis=1)
nonzero_mask = df[COLS_CONC].sum(axis=1) > 0

print('=== Vérification mathématique ===')
print(f'Échantillons avec concentration totale > 0 : {nonzero_mask.sum()}')
verif = np.allclose(sum_squares[nonzero_mask], 1.0, atol=1e-10)
print(f'sum_j(y\'_ij)² = 1 pour tous les échantillons non nuls : {verif}')
print(f'Écart max observé : {(sum_squares[nonzero_mask] - 1).abs().max():.2e}')

# Visualisation : avant vs après transformation pour les 3 groupes les plus fréquents
top3 = (df[COLS_CONC] > 0).mean().nlargest(3).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, col in enumerate(top3):
    data_raw = df[col][df[col] > 0]
    data_hell = df_hellinger[col][df_hellinger[col] > 0]

    axes[0, i].hist(np.log10(data_raw + 1e-12), bins=35, color='salmon', alpha=0.8, edgecolor='white')
    axes[0, i].set_title(f'{col}\nAvant (log₁₀)', fontsize=9, fontweight='bold')
    axes[0, i].set_xlabel('log₁₀(concentration)', fontsize=8)

    axes[1, i].hist(data_hell, bins=35, color='mediumseagreen', alpha=0.8, edgecolor='white')
    axes[1, i].set_title(f'{col}\nAprès (Hellinger)', fontsize=9, fontweight='bold')
    axes[1, i].set_xlabel('Valeur transformée', fontsize=8)

plt.suptitle('Effet de la transformation de Hellinger\n(3 groupes les plus fréquents)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_06_hellinger_transformation.png', dpi=120, bbox_inches='tight')
plt.show()

## 2.2 Analyse en Composantes Principales (ACP)

L'ACP est appliquée sur les données transformées par Hellinger. L'algorithme :
1. Centre automatiquement les données (soustraction de la moyenne par variable)
2. Calcule les composantes principales (vecteurs propres de la matrice de covariance)
3. Projette les données dans le nouvel espace orthogonal

In [ ]:
# === ACP sur données Hellinger ===

X_hell = df_hellinger.values

# ACP complète
pca = PCA(random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_hell)

exp_var  = pca.explained_variance_ratio_
cum_var  = np.cumsum(exp_var)
n_total  = len(exp_var)

# Nombre de composantes pour différents seuils
n_80 = int(np.argmax(cum_var >= 0.80)) + 1
n_90 = int(np.argmax(cum_var >= 0.90)) + 1
n_95 = int(np.argmax(cum_var >= 0.95)) + 1

print('=== Résultats de l\'ACP ===')
print(f'Nombre total de composantes : {n_total}')
print(f'Variance expliquée par les 2 premières composantes : {cum_var[1]:.1%}')
print(f'Variance expliquée par les 3 premières composantes : {cum_var[2]:.1%}')
print(f'Composantes pour 80% de variance : {n_80}')
print(f'Composantes pour 90% de variance : {n_90}')
print(f'Composantes pour 95% de variance : {n_95}')
print(f'\nVariance expliquée par composante (10 premières) :')
for i in range(min(10, n_total)):
    bar = '#' * int(exp_var[i] * 100)
    print(f'  PC{i+1:2d} : {exp_var[i]:.3%}  (cumulée : {cum_var[i]:.3%})  {bar}')

In [ ]:
# === Scree plot (éboulis des valeurs propres) ===

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

n_show = min(20, n_total)
x_ax = np.arange(1, n_show + 1)

# --- Variance individuelle ---
axes[0].bar(x_ax, exp_var[:n_show] * 100,
            color='steelblue', alpha=0.8, edgecolor='white', label='Variance expliquée (%)')
axes[0].plot(x_ax, exp_var[:n_show] * 100, 'o-', color='darkblue', markersize=5)
axes[0].set_xlabel('Numéro de composante', fontweight='bold')
axes[0].set_ylabel('Variance expliquée (%)', fontweight='bold')
axes[0].set_title('Scree plot — Variance par composante', fontweight='bold')
axes[0].set_xticks(x_ax)

# --- Variance cumulée ---
axes[1].plot(x_ax, cum_var[:n_show] * 100, 's-', color='steelblue', markersize=6, linewidth=2)
axes[1].fill_between(x_ax, 0, cum_var[:n_show] * 100, alpha=0.2, color='steelblue')

# Lignes de référence (80%, 90%, 95%)
for thresh, color, label in [(80, 'green', '80%'), (90, 'orange', '90%'), (95, 'red', '95%')]:
    axes[1].axhline(thresh, linestyle='--', color=color, alpha=0.7, linewidth=1)
    axes[1].text(n_show - 0.5, thresh + 0.5, label, color=color, fontsize=9, ha='right')

axes[1].set_xlabel('Nombre de composantes', fontweight='bold')
axes[1].set_ylabel('Variance cumulée (%)', fontweight='bold')
axes[1].set_title('Variance cumulée par composante', fontweight='bold')
axes[1].set_xticks(x_ax)
axes[1].set_ylim(0, 105)
axes[1].grid(True, alpha=0.4)

plt.suptitle('ACP — Analyse de la variance expliquée', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_07_pca_screeplot.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# === Visualisation des individus dans le plan factoriel (PC1 x PC2) ===

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

pc1_var = exp_var[0] * 100
pc2_var = exp_var[1] * 100

# --- Coloré par couche ---
layer_values = df['layer'].values
unique_layers = sorted(df['layer'].dropna().unique())
le = LabelEncoder()
layer_encoded = le.fit_transform(pd.Series(layer_values).fillna('unknown'))
colors_map = plt.cm.Set1(np.linspace(0, 0.8, len(unique_layers)))

for k, layer in enumerate(unique_layers):
    mask = layer_values == layer
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    s=10, alpha=0.5, color=colors_map[k], label=layer)

axes[0].set_xlabel(f'PC1 ({pc1_var:.1f}% de variance)', fontweight='bold')
axes[0].set_ylabel(f'PC2 ({pc2_var:.1f}% de variance)', fontweight='bold')
axes[0].set_title('Projection des individus — couleur par couche', fontweight='bold')
axes[0].legend(title='Couche', markerscale=2, fontsize=9)
axes[0].axhline(0, color='gray', linewidth=0.5, linestyle='--')
axes[0].axvline(0, color='gray', linewidth=0.5, linestyle='--')

# --- Coloré par année ---
sc = axes[1].scatter(X_pca[:, 0], X_pca[:, 1],
                     s=10, alpha=0.5,
                     c=df['year'], cmap='viridis')
plt.colorbar(sc, ax=axes[1], label='Année')
axes[1].set_xlabel(f'PC1 ({pc1_var:.1f}% de variance)', fontweight='bold')
axes[1].set_ylabel(f'PC2 ({pc2_var:.1f}% de variance)', fontweight='bold')
axes[1].set_title('Projection des individus — couleur par année', fontweight='bold')
axes[1].axhline(0, color='gray', linewidth=0.5, linestyle='--')
axes[1].axvline(0, color='gray', linewidth=0.5, linestyle='--')

plt.suptitle(f'ACP — Plan factoriel PC1 × PC2 ({pc1_var + pc2_var:.1f}% de variance)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_08_pca_individus.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# === Cercle des corrélations (variables) ===

# Calcul des coordonnées des variables (loadings)
loadings = pca.components_[:2].T  # shape (n_variables, 2)
# Multiplier par sqrt(valeurs propres) pour obtenir les corrélations
loadings_scaled = loadings * np.sqrt(pca.explained_variance_[:2])

fig, ax = plt.subplots(figsize=(10, 10))

# Cercle unitaire
theta = np.linspace(0, 2 * np.pi, 300)
ax.plot(np.cos(theta), np.sin(theta), 'k-', linewidth=0.8, alpha=0.4)
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')

# Flèches et labels
colors_species = plt.cm.tab20(np.linspace(0, 1, len(COLS_CONC)))
for i, (col, (x, y)) in enumerate(zip(COLS_CONC, loadings_scaled)):
    ax.annotate('', xy=(x, y), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=colors_species[i], lw=1.5))
    # Décalage du label
    offset = 0.04
    ax.text(x + np.sign(x) * offset, y + np.sign(y) * offset,
            col, fontsize=8, ha='center', va='center', color=colors_species[i], fontweight='bold')

ax.set_xlim(-1.15, 1.15)
ax.set_ylim(-1.15, 1.15)
ax.set_xlabel(f'PC1 ({pc1_var:.1f}%)', fontweight='bold', fontsize=12)
ax.set_ylabel(f'PC2 ({pc2_var:.1f}%)', fontweight='bold', fontsize=12)
ax.set_title('Cercle des corrélations — ACP (Hellinger)', fontsize=14, fontweight='bold')
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('fig_09_pca_cercle_correlations.png', dpi=120, bbox_inches='tight')
plt.show()

## 2.3 Classification Baseline : K-means

La classification K-means est appliquée sur les **composantes principales** conservées (seuil de 80% de variance expliquée). Cette approche est cohérente avec la méthodologie de l'article de référence :
- Réduction du bruit en ignorant les composantes de faible variance
- Préservation de la structure principale des données
- Distance euclidienne dans le plan des composantes principales

La **méthode du coude** (inertie intra-clusters) et le **score de silhouette** sont utilisés pour déterminer le nombre optimal de clusters $k$.

In [ ]:
# === Détermination du nombre optimal de clusters ===

# Données pour le clustering : n_80 premières composantes principales
X_clust = X_pca[:, :n_80]
print(f'Données de clustering : {X_clust.shape} ({n_80} composantes → {cum_var[n_80-1]:.1%} de variance)')

K_range = range(2, 11)
inertias = []
silhouettes = []
db_scores = []
ch_scores = []

print('Calcul des métriques en cours...')
for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10, max_iter=300)
    labels = km.fit_predict(X_clust)
    inertias.append(km.inertia_)
    # Silhouette sur un sous-échantillon si les données sont grandes
    n_sample = min(5000, len(labels))
    silhouettes.append(silhouette_score(X_clust, labels, sample_size=n_sample, random_state=RANDOM_STATE))
    db_scores.append(davies_bouldin_score(X_clust, labels))
    ch_scores.append(calinski_harabasz_score(X_clust, labels))
    print(f'  k={k} | Inertie: {km.inertia_:.2f} | Silhouette: {silhouettes[-1]:.4f} | DB: {db_scores[-1]:.4f}')

# Tableau récapitulatif
metrics_df = pd.DataFrame({
    'k': list(K_range),
    'Inertie': inertias,
    'Silhouette (↑)': silhouettes,
    'Davies-Bouldin (↓)': db_scores,
    'Calinski-Harabasz (↑)': ch_scores
}).set_index('k')
print('\n=== Métriques de clustering ===')
print(metrics_df.round(4).to_string())

In [ ]:
# === Visualisation des métriques ===

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
K_list = list(K_range)

# Inertie (méthode du coude)
axes[0, 0].plot(K_list, inertias, 's-', color='steelblue', markersize=8, linewidth=2)
axes[0, 0].fill_between(K_list, inertias, alpha=0.1, color='steelblue')
axes[0, 0].set_xlabel('Nombre de clusters k', fontweight='bold')
axes[0, 0].set_ylabel('Inertie intra-clusters', fontweight='bold')
axes[0, 0].set_title('Méthode du coude (Elbow)', fontweight='bold')
axes[0, 0].set_xticks(K_list)
axes[0, 0].grid(True, alpha=0.4)

# Score de silhouette
best_sil_k = K_list[np.argmax(silhouettes)]
axes[0, 1].plot(K_list, silhouettes, 'D-', color='coral', markersize=8, linewidth=2)
axes[0, 1].axvline(best_sil_k, color='red', linestyle='--', alpha=0.7,
                    label=f'Optimal k={best_sil_k} (silhouette={max(silhouettes):.4f})')
axes[0, 1].set_xlabel('Nombre de clusters k', fontweight='bold')
axes[0, 1].set_ylabel('Score de Silhouette (plus élevé = mieux)', fontweight='bold')
axes[0, 1].set_title('Score de Silhouette', fontweight='bold')
axes[0, 1].set_xticks(K_list)
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, alpha=0.4)

# Davies-Bouldin
best_db_k = K_list[np.argmin(db_scores)]
axes[1, 0].plot(K_list, db_scores, '^-', color='mediumseagreen', markersize=8, linewidth=2)
axes[1, 0].axvline(best_db_k, color='green', linestyle='--', alpha=0.7,
                    label=f'Optimal k={best_db_k} (DB={min(db_scores):.4f})')
axes[1, 0].set_xlabel('Nombre de clusters k', fontweight='bold')
axes[1, 0].set_ylabel('Indice Davies-Bouldin (plus bas = mieux)', fontweight='bold')
axes[1, 0].set_title('Indice Davies-Bouldin', fontweight='bold')
axes[1, 0].set_xticks(K_list)
axes[1, 0].legend(fontsize=9)
axes[1, 0].grid(True, alpha=0.4)

# Calinski-Harabasz
best_ch_k = K_list[np.argmax(ch_scores)]
axes[1, 1].plot(K_list, ch_scores, 'o-', color='mediumpurple', markersize=8, linewidth=2)
axes[1, 1].axvline(best_ch_k, color='purple', linestyle='--', alpha=0.7,
                    label=f'Optimal k={best_ch_k} (CH={max(ch_scores):.1f})')
axes[1, 1].set_xlabel('Nombre de clusters k', fontweight='bold')
axes[1, 1].set_ylabel('Indice Calinski-Harabasz (plus élevé = mieux)', fontweight='bold')
axes[1, 1].set_title('Indice Calinski-Harabasz', fontweight='bold')
axes[1, 1].set_xticks(K_list)
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True, alpha=0.4)

plt.suptitle('Détermination du nombre optimal de clusters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_10_kmeans_metriques.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Recommandations des métriques :')
print(f'  Score de Silhouette → k = {best_sil_k}')
print(f'  Davies-Bouldin       → k = {best_db_k}')
print(f'  Calinski-Harabasz    → k = {best_ch_k}')

In [ ]:
# === Application du K-means avec le k optimal ===

# Choisir le k basé sur le score de silhouette
K_OPTIMAL = best_sil_k
print(f'Nombre de clusters retenu : k = {K_OPTIMAL}')
print('(Basé sur le score de silhouette — cohérent avec la méthode du coude)')

kmeans_final = KMeans(n_clusters=K_OPTIMAL, random_state=RANDOM_STATE, n_init=20, max_iter=500)
df['cluster'] = kmeans_final.fit_predict(X_clust)

# Métriques finales
labels_final = df['cluster'].values
sil_final = silhouette_score(X_clust, labels_final, sample_size=min(5000, len(labels_final)), random_state=RANDOM_STATE)
db_final  = davies_bouldin_score(X_clust, labels_final)
ch_final  = calinski_harabasz_score(X_clust, labels_final)

print(f'\n=== Métriques finales (k={K_OPTIMAL}) ===')
print(f'  Score de Silhouette    : {sil_final:.4f}  (0–1, plus élevé = mieux)')
print(f'  Indice Davies-Bouldin  : {db_final:.4f}  (plus bas = mieux)')
print(f'  Indice Calinski-Harabasz: {ch_final:.2f} (plus élevé = mieux)')

# Taille des clusters
cluster_sizes = df['cluster'].value_counts().sort_index()
print(f'\n=== Taille des clusters ===')
for c, n in cluster_sizes.items():
    pct = n / len(df) * 100
    print(f'  Cluster {c} : {n:5d} observations ({pct:.1f}%)')

In [ ]:
# === Visualisation des clusters ===

cluster_colors = plt.cm.Set1(np.linspace(0, 0.9, K_OPTIMAL))
cluster_names = [f'Communauté {i+1}' for i in range(K_OPTIMAL)]

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# --- 1. Plan factoriel PC1 x PC2 ---
for c in range(K_OPTIMAL):
    mask = labels_final == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    s=15, alpha=0.6, color=cluster_colors[c], label=cluster_names[c])

# Centroïdes des clusters
centroids_pca = np.array([X_pca[labels_final == c, :2].mean(axis=0) for c in range(K_OPTIMAL)])
for c in range(K_OPTIMAL):
    axes[0].scatter(*centroids_pca[c], s=200, marker='*', color=cluster_colors[c],
                    edgecolors='black', linewidths=1.5, zorder=10)

axes[0].set_xlabel(f'PC1 ({pc1_var:.1f}%)', fontweight='bold')
axes[0].set_ylabel(f'PC2 ({pc2_var:.1f}%)', fontweight='bold')
axes[0].set_title('Clusters dans le plan PC1 × PC2\n(★ = centroïde)', fontweight='bold')
axes[0].legend(fontsize=8, markerscale=1.5)
axes[0].axhline(0, color='gray', linewidth=0.4, linestyle='--')
axes[0].axvline(0, color='gray', linewidth=0.4, linestyle='--')

# --- 2. Plan factoriel PC1 x PC3 ---
pc3_var = exp_var[2] * 100
for c in range(K_OPTIMAL):
    mask = labels_final == c
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 2],
                    s=15, alpha=0.6, color=cluster_colors[c], label=cluster_names[c])

axes[1].set_xlabel(f'PC1 ({pc1_var:.1f}%)', fontweight='bold')
axes[1].set_ylabel(f'PC3 ({pc3_var:.1f}%)', fontweight='bold')
axes[1].set_title('Clusters dans le plan PC1 × PC3', fontweight='bold')
axes[1].legend(fontsize=8, markerscale=1.5)
axes[1].axhline(0, color='gray', linewidth=0.4, linestyle='--')
axes[1].axvline(0, color='gray', linewidth=0.4, linestyle='--')

# --- 3. Distribution géographique des clusters ---
for c in range(K_OPTIMAL):
    mask = labels_final == c
    axes[2].scatter(df.loc[mask, 'lon'], df.loc[mask, 'lat'],
                    s=15, alpha=0.5, color=cluster_colors[c], label=cluster_names[c])

axes[2].set_xlabel('Longitude (°)', fontweight='bold')
axes[2].set_ylabel('Latitude (°)', fontweight='bold')
axes[2].set_title('Distribution géographique des clusters', fontweight='bold')
axes[2].legend(fontsize=8, markerscale=1.5)
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Classification K-means — {K_OPTIMAL} communautés planctoniques (Baseline)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_11_kmeans_clusters.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# === Caractérisation biologique des clusters ===

# Profil de concentration moyen par cluster (données Hellinger)
df_hell_clusters = df_hellinger.copy()
df_hell_clusters['cluster'] = labels_final

cluster_profiles = df_hell_clusters.groupby('cluster')[COLS_CONC].mean()

# Normalisation des profils pour la heatmap (0–1 par groupe)
profile_norm = cluster_profiles.div(cluster_profiles.max(axis=0).replace(0, 1), axis=1)

# Heatmap des profils
fig, axes = plt.subplots(1, 2, figsize=(22, 6))

# Profils normalisés
sns.heatmap(profile_norm.T, annot=False, cmap='YlOrRd', ax=axes[0],
            xticklabels=[f'C{i+1}' for i in range(K_OPTIMAL)],
            linewidths=0.3, linecolor='white', vmin=0, vmax=1)
axes[0].set_title('Profils planctoniques par cluster\n(valeurs Hellinger normalisées 0–1)',
                  fontweight='bold')
axes[0].set_xlabel('Cluster', fontweight='bold')
axes[0].set_ylabel('Groupe taxonomique', fontweight='bold')
axes[0].tick_params(axis='y', labelsize=8)

# Caractérisation par variables contextuelles
context_stats = df.groupby('cluster').agg(
    n_obs=('psampleid', 'count'),
    pct_day=('day_night', lambda x: (x == 'day').mean() * 100),
    lat_mean=('lat', 'mean'),
    lon_mean=('lon', 'mean')
).round(2)

# Caractérisation couche
layer_dist = df.groupby(['cluster', 'layer']).size().unstack(fill_value=0)
layer_pct = layer_dist.div(layer_dist.sum(axis=1), axis=0) * 100

sns.heatmap(layer_pct.T, annot=True, fmt='.0f', cmap='Blues', ax=axes[1],
            xticklabels=[f'C{i+1}' for i in range(K_OPTIMAL)],
            linewidths=0.3, linecolor='white')
axes[1].set_title('Distribution par couche océanique (%)\npar cluster', fontweight='bold')
axes[1].set_xlabel('Cluster', fontweight='bold')
axes[1].set_ylabel('Couche océanique', fontweight='bold')

plt.suptitle('Caractérisation biologique et contextuelle des clusters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_12_cluster_profils.png', dpi=120, bbox_inches='tight')
plt.show()

print('=== Statistiques contextuelles par cluster ===')
print(context_stats.to_string())

In [ ]:
# === Caractérisation environnementale des clusters ===

key_env_char = ['temp', 'sal', 'chla', 'oxy_ml', 'mld', 'ze', 'poc']
key_labels_char = ['Temp. (°C)', 'Salinité', 'Chla (mg/m³)', 'O₂ (ml/L)', 'MLD (m)', 'Ze (m)', 'POC (mg/m³)']

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i, (var, label) in enumerate(zip(key_env_char, key_labels_char)):
    data_by_cluster = [df[df['cluster'] == c][var].dropna().values for c in range(K_OPTIMAL)]
    bp = axes[i].boxplot(data_by_cluster,
                         patch_artist=True,
                         medianprops={'color': 'black', 'linewidth': 2},
                         flierprops={'marker': '.', 'markersize': 3, 'alpha': 0.3})
    for patch, color in zip(bp['boxes'], cluster_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    axes[i].set_title(label, fontweight='bold', fontsize=10)
    axes[i].set_xticklabels([f'C{c+1}' for c in range(K_OPTIMAL)], fontsize=9)
    axes[i].set_xlabel('Cluster', fontsize=9)
    axes[i].grid(True, alpha=0.3, axis='y')

# Masquer le 8ème axe (vide)
axes[7].set_visible(False)

plt.suptitle('Distribution des variables environnementales par cluster',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_13_cluster_env.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# === Tableau récapitulatif des caractéristiques environnementales par cluster ===

env_summary_vars = ['temp', 'sal', 'chla', 'oxy_ml', 'mld', 'ze', 'poc', 'par']
env_summary_labels = ['Temp. (°C)', 'Salinité', 'Chla', 'O2 (ml/L)', 'MLD (m)', 'Ze (m)', 'POC', 'PAR']

rows = []
for c in range(K_OPTIMAL):
    sub = df[df['cluster'] == c]
    row = {'Cluster': f'C{c+1} (n={len(sub)})'}
    for var, lbl in zip(env_summary_vars, env_summary_labels):
        vals = sub[var].dropna()
        row[lbl] = f'{vals.mean():.2f} ± {vals.std():.2f}' if len(vals) > 0 else 'NA'
    rows.append(row)

env_table = pd.DataFrame(rows).set_index('Cluster')
print('=== Caractéristiques environnementales moyennes par cluster (moyenne ± écart-type) ===')
print(env_table.to_string())

# Top 3 groupes les plus contributeurs par cluster
print('\n=== Top 3 groupes planctoniques par cluster (Hellinger) ===')
for c in range(K_OPTIMAL):
    top3 = cluster_profiles.loc[c].nlargest(3)
    print(f'  Cluster {c+1} : ' + ', '.join([f'{g} ({v:.4f})' for g, v in top3.items()]))

## 2.4 Interprétation des Résultats

### Synthèse de la baseline

La reproduction de l'analyse de Panaïotis et al. (2023) a permis d'identifier plusieurs **communautés planctoniques** distinctes à partir des données de concentration transformées par Hellinger, réduites par ACP, puis classifiées par K-means.

### Qualité de la classification

- **Transformation de Hellinger** : Vérifiée mathématiquement ($\sum_j y'^2_{ij} = 1$). Elle permet de comparer les **profils** planctoniques indépendamment des abondances absolues.
- **ACP** : Les premières composantes capturent une part significative de la variance. Le plan factoriel PC1 × PC2 révèle une structuration des données.
- **K-means** : Les métriques (silhouette, Davies-Bouldin, Calinski-Harabasz) convergent vers un nombre de clusters cohérent. Les communautés identifiées présentent des profils environnementaux et taxonomiques distincts.

### Limites de la baseline

1. **Linéarité** : L'ACP ne capture que les relations linéaires : des structures non-linéaires pourraient être manquées
2. **Forme des clusters** : K-means suppose des clusters sphériques de même taille : une hypothèse potentiellement restrictive
3. **Sensibilité à k** : Le choix du nombre de clusters influence fortement les résultats
4. **Variables environnementales** : Non intégrées dans la classification elle-même (utilisées uniquement pour la caractérisation a posteriori)

### Perspectives (Jalons 2 et 3)

Ces limites motivent l'exploration de :
- **Nouvelles réductions dimensionnelles** : AFC, Isomap, LLE, MDS (Jalon 2)
- **Nouveaux algorithmes de clustering** : DBSCAN, HDBSCAN, Spectral Clustering, GMM (Jalon 3)

---

## Références bibliographiques

- **Panaïotis et al. (2023)** : Jeu de données SEANOE, référence 103673
- **Hellinger, E. (1909)** : Neue Begründung der Theorie quadratischer Formen von unendlichvielen Veränderlichen
- **Pearson, K. (1901)** : On lines and planes of closest fit to systems of points in space. *Philosophical Magazine*, 2(11)
- **Legendre, P. & Gallagher, E.D. (2001)** : Ecologically meaningful transformations for ordination of species data. *Oecologia*, 129(2)
- **MacQueen, J. (1967)** : Some methods for classification and analysis of multivariate observations. *Proceedings of the 5th Berkeley Symposium*

<a id="sauvegarde"></a>

---

## 3. Sauvegarde des résultats


In [ ]:
# === Sauvegarde des résultats pour Jalon 2 et 3 ===
import pickle

jalon1_outputs = {
    'df': df,
    'df_hellinger': df_hellinger,
    'COLS_CONC': COLS_CONC,
    'COLS_ENV': COLS_ENV,
    'X_pca': X_pca,
    'n_80': n_80,
    'pca': pca,
    'labels_final': labels_final,
    'best_sil_k': best_sil_k,
    'K_OPTIMAL': K_OPTIMAL,
    'unique_layers': unique_layers,
}
with open('data/jalon1_outputs.pkl', 'wb') as f:
    pickle.dump(jalon1_outputs, f)
print("Résultats Jalon 1 sauvegardés → data/jalon1_outputs.pkl")


<a id="conclusions"></a>

---

## 4. Conclusions & Perspectives

### Synthèse du Jalon 1

Ce premier jalon a permis d'établir une base solide pour l'analyse des communautés planctoniques.
La transformation de Hellinger normalise efficacement les données de comptage sur-dispersées,
et l'ACP révèle que les premières composantes capturent principalement le gradient épipelagique/mésopelagique.
La classification K-means baseline offre un point de comparaison pour les méthodes avancées des jalons suivants.

### Points clés
| Élément | Résultat |
|---|---|
| Transformation | Hellinger (stabilisation de variance) |
| Réduction de dimension | ACP : composantes pour 80 % de variance |
| Clustering baseline | K-means, k optimal par score de silhouette |

### Perspectives : Jalon 2
Le notebook `02_representations.ipynb` explorera des méthodes de représentation non-linéaires
(AFC, Isomap, LLE, MDS) pour capturer des structures que l'ACP linéaire ne peut pas révéler.
